In [1]:
%load_ext autoreload
%autoreload 2


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap

import celltrip


mpl.rcParams['pdf.fonttype'] = 42


# Load Data

In [3]:
# Read data
adata, = celltrip.utility.processing.read_adatas('s3://nkalafut-celltrip/NHP/top5k_expression.h5ad')

# Correct variables
adata.obs['age_orig'] = adata.obs['age']
adata.obs['age'] = adata.obs['age'].apply(lambda x: {  # Translate ages (TODO: Verify that this is correct)
    k: v for k, v in zip(
        ['E87', 'E96', 'E101', 'E104', 'E112', 'E127', 'E150', 'E155'],
        (85, 95, 100, 105, 110, 125, 145, 155))}[x])
adata.obs['subtype_orig'] = adata.obs['subtype']
adata.obs['subtype'] = adata.obs['subtype'].map(lambda s: ' '.join(s.split(' ')[:2]), na_action='ignore')  # Shorten subtypes

# Normalize data for visualization
sim_data = adata.X[:]
sim_data_norm = 10_000 * sim_data / sim_data.sum(axis=1)


In [ ]:
# Load manager
prefix, training_step = 's3://nkalafut-celltrip/checkpoints/NHP-260324', 800
manager = celltrip.manager.BasicManager(
    policy_fname=f'{prefix}-{training_step:04}.weights',
    preprocessing_fname=f'{prefix}.pre',
    mask_fname=f'{prefix}.mask',
    device='cuda')

# Use validation, unpartitioned data
manager.set_modalities([adata[~adata.obs['Train']]], suppress_warning=True)


In [ ]:
# Plot
orig_red = umap.UMAP()
orig_emb = orig_red.fit_transform(adata[~adata.obs['Train']].X[:])
df = pd.DataFrame(orig_emb, columns=['UMAP1', 'UMAP2'], index=adata.obs_names)
df['age'] = adata.obs['age']
df['subtype'] = adata.obs['subtype']
df['sample'] = adata.obs['sample']
sns.scatterplot(data=df, x='UMAP1', y='UMAP2', hue='subtype', style='age')


# Simulate

In [ ]:
# Simulate to steady state
manager.reset_env()
manager.simulate(skip_time=512., impute=False)
manager.save_state('steady')
steady_gex, = manager.get_state()

In [ ]:
# Plot
steady_red = umap.UMAP()
steady_emb = steady_red.fit_transform(steady_gex)
df = pd.DataFrame(steady_emb, columns=['UMAP1', 'UMAP2'], index=adata.obs_names)
df['age'] = adata.obs['age']
df['subtype'] = adata.obs['subtype']
df['sample'] = adata.obs['sample']
sns.scatterplot(data=df, x='UMAP1', y='UMAP2', hue='subtype', style='age')


# Knockdown

In [ ]:
# Get control
manager.reset_env()
manager.load_state('steady')
manager.simulate_perturbation(skip_time=512., time=128., impute=False)
control_gex, = manager.get_state()


In [ ]:
# Plot
control_red = umap.UMAP()
control_emb = control_red.fit_transform(control_gex)
df = pd.DataFrame(control_emb, columns=['UMAP1', 'UMAP2'], index=adata.obs_names)
df['age'] = adata.obs['age']
df['subtype'] = adata.obs['subtype']
df['sample'] = adata.obs['sample']
sns.scatterplot(data=df, x='UMAP1', y='UMAP2', hue='subtype', style='age')


In [ ]:
# Perform knockdown
manager.reset_env()
manager.load_state('steady')
manager.add_perturbation(['SOME_FEATURE'], modality=0, feature_targets=0)  # TODO
manager.simulate_perturbation(skip_time=512., time=128., impute=False)
pert_gex, = manager.get_state()

In [ ]:
# Plot
control_emb = control_red.transform(pert_gex)
df = pd.DataFrame(control_emb, columns=['UMAP1', 'UMAP2'], index=adata.obs_names)
df['age'] = adata.obs['age']
df['subtype'] = adata.obs['subtype']
df['sample'] = adata.obs['sample']
sns.scatterplot(data=df, x='UMAP1', y='UMAP2', hue='subtype', style='age')
